# Churn con scikit-learn y MLflow en Databricks

Este notebook toma el pipeline de churn de la Actividad 1 y lo conecta con MLflow.

Se usan dos funcionalidades de MLflow:

1. **Experiments:** guarda los parámetros, las métricas y el modelo de cada corrida.
2. **Models:** registra el modelo en Unity Catalog con un alias y lo carga por ese alias.

El objetivo de negocio no cambia: identificar clientes con alta probabilidad de cancelar el servicio.

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Configuración de MLflow

El modelo se guarda en Unity Catalog con un nombre de tres niveles: catálogo, esquema y modelo.
Hay que cambiar `CATALOG` y `SCHEMA` si el modelo debe quedar en otro lugar.

In [ ]:
CATALOG = "workspace"
SCHEMA = "default"
MODEL_ALIAS = "champion"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.churn_gradient_boosting"

mlflow.set_registry_uri("databricks-uc")
print("Modelo:", MODEL_NAME)

## Datos

Es el dataset Telco Customer Churn. Se lee desde la misma URL pública que usa el proyecto original.
Si el workspace bloquea el acceso a internet, hay que subir el CSV y poner su ruta en `DATA_PATH`.

In [ ]:
SOURCE_URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)
DATA_PATH = None

raw_df = pd.read_csv(DATA_PATH or SOURCE_URL)
print("Filas y columnas:", raw_df.shape)

## Pipeline de scikit-learn

Es el mismo pipeline de la Actividad 1:

1. **Extracción:** selecciona las columnas y convierte `TotalCharges` a número.
2. **Filtrado:** quita clientes duplicados y valores poco creíbles.
3. **Preprocesamiento:** imputa, escala y codifica las columnas.
4. **Modelo:** `GradientBoostingClassifier` con los hiperparámetros ganadores de la Actividad 1.

In [ ]:
ID_COLUMN = "customerID"
TARGET_COLUMN = "Churn"
NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
CATEGORICAL_FEATURES = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod",
]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
MIN_PLAUSIBLE_TENURE = 0
MAX_PLAUSIBLE_TENURE = 100


class DataExtractor(BaseEstimator, TransformerMixin):
    """Selecciona las columnas relevantes y corrige el tipo de TotalCharges."""

    def __init__(self, columns=None, include_target=True):
        self.columns = columns
        self.include_target = include_target

    def fit(self, X, y=None):
        base_columns = self.columns or FEATURE_COLUMNS
        self._columns = [ID_COLUMN] + base_columns
        if self.include_target and TARGET_COLUMN not in self._columns:
            self._columns = self._columns + [TARGET_COLUMN]
        return self

    def transform(self, X):
        missing = [c for c in self._columns if c not in X.columns]
        if missing:
            raise KeyError(f"Faltan columnas esperadas en el origen: {missing}")
        result = X[self._columns].copy()
        result["TotalCharges"] = pd.to_numeric(result["TotalCharges"], errors="coerce")
        return result


class DataFilter(BaseEstimator, TransformerMixin):
    """Elimina clientes duplicados y filas con valores poco creibles."""

    def __init__(self, min_tenure=MIN_PLAUSIBLE_TENURE, max_tenure=MAX_PLAUSIBLE_TENURE):
        self.min_tenure = min_tenure
        self.max_tenure = max_tenure

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        result = X.drop_duplicates(subset=[ID_COLUMN])
        tenure_en_rango = result["tenure"].between(self.min_tenure, self.max_tenure)
        result = result.loc[tenure_en_rango]
        return result.drop(columns=[ID_COLUMN]).copy()


def build_cleaning_pipeline():
    return Pipeline(steps=[("extraction", DataExtractor()), ("filtering", DataFilter())])


def build_preprocessing_pipeline():
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer(transformers=[
        ("numeric", numeric_transformer, NUMERIC_FEATURES),
        ("categorical", categorical_transformer, CATEGORICAL_FEATURES),
    ])
    return Pipeline(steps=[("preprocessing", preprocessor)])


def split_dataset(clean_df, test_size=0.2, random_state=42):
    X = clean_df[FEATURE_COLUMNS]
    y = clean_df[TARGET_COLUMN].map({"Yes": 1, "No": 0})
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)


# Hiperparametros ganadores de la busqueda hecha en la Actividad 1.
MODEL_PARAMS = {"learning_rate": 0.1, "max_depth": 2, "n_estimators": 100, "random_state": 42}


def build_model_pipeline():
    return Pipeline(steps=[
        ("preprocessing", build_preprocessing_pipeline()),
        ("classifier", GradientBoostingClassifier(**MODEL_PARAMS)),
    ])

In [ ]:
clean_df = build_cleaning_pipeline().fit_transform(raw_df)
X_train, X_test, y_train, y_test = split_dataset(clean_df)

print("Datos limpios:", clean_df.shape)
print("Train:", X_train.shape, "Test:", X_test.shape)

## Experiments: entrenar y registrar la corrida

`autolog` guarda los parámetros del pipeline y las métricas de entrenamiento.
Las métricas de prueba se registran a mano.
El modelo se registra con su firma, que documenta las columnas de entrada y salida.

In [ ]:
mlflow.sklearn.autolog(log_models=False)

with mlflow.start_run(run_name="churn-gradient-boosting") as run:
    model = build_model_pipeline()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    mlflow.log_metrics(
        {
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_precision": precision_score(y_test, y_pred),
            "test_recall": recall_score(y_test, y_pred),
            "test_f1": f1_score(y_test, y_pred),
            "test_roc_auc": roc_auc_score(y_test, y_proba),
        }
    )
    mlflow.set_tags({"dataset": "telco_customer_churn", "etapa_crisp_dm": "modeling"})

    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        signature=signature,
        input_example=X_train.head(3),
        # El formato por defecto de MLflow rechaza los arboles de Gradient Boosting.
        serialization_format="cloudpickle",
        registered_model_name=MODEL_NAME,
    )
    print("Run ID:", run.info.run_id)

## Models: alias champion y carga del modelo

El alias `champion` apunta a una versión del modelo.
Quien consume el modelo lo carga por alias y no necesita saber el número de versión.

In [ ]:
client = MlflowClient()
versions = client.search_model_versions(f"name = '{MODEL_NAME}'")
latest_version = max(versions, key=lambda version: int(version.version)).version
client.set_registered_model_alias(MODEL_NAME, MODEL_ALIAS, latest_version)
print(f"El alias {MODEL_ALIAS} apunta a la version {latest_version}")

champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@{MODEL_ALIAS}")
pd.DataFrame({"real": y_test.head(5).values, "predicho": champion.predict(X_test.head(5))})

## Dónde ver los resultados en Databricks

1. **Experiments:** en el panel lateral del notebook o en la sección Experiments del workspace. Ahí están los parámetros, las métricas y el modelo de la corrida.
2. **Models:** en Catalog, dentro del esquema indicado, en la sección Models. Ahí están las versiones y el alias.